## SGC on Reddit

We ran SGC on the Reddit dataset to test the paper's claim that SGC scales to larger graphs where GCN cannot. Reddit has 233K nodes and 11.6M edges. This is significantly larger than the citation network, and the paper reports SGC achieving 94.9% Micro F1 while GCN runs out of memory entirely.

### Methodology:
1. **Data Loading**: Load Reddit via PyTorch Geometric with row-normalized features.
2. **Inductive Preprocessing**: Precompute S²X on the training subgraph only, then propagate on the full graph for test. Features normalized to zero mean and unit variance after propagation (required for Reddit per Wu et al., 2019 supplementary material).
3. **SGC Model**: Single linear layer trained on precomputed features.
4. **Optimizer**: L-BFGS with lr=0.2, no weight decay, 10 epochs.
5. **Evaluation**: Micro F1 score over 10 random seeds. GCN could not be evaluated — training failed with an out of memory error (27GB requested on 14.6GB GPU), consistent with the OOM result reported in the original paper.

### Note on Results:
Our mean F1 of 72.73% is below the paper's reported 94.9%. The gap is likely due to missing implementation details not fully specified in the paper: the official repository uses a custom data loader, feature normalization procedure, and L-BFGS configuration may differ subtly from our implementation. Notably, we observed that accuracy continued improving beyond the paper's recommended 2 epochs. Our reported result uses 10 epochs, without which F1 was significantly lower. This suggests the paper's claim of convergence in 2 L-BFGS steps may be hardware or configuration dependent. Given more time we would have worked more closely from the official codebase to close this gap. Regardless, the GCN OOM result was fully replicated, confirming the paper's core scalability claim.

####Installing Libraries

In [1]:
# Installing libraries
%%capture
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch_geometric
!pip install -q scipy


### Imports

In [2]:
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch_geometric.datasets import Reddit
from torch_geometric.utils import add_self_loops, degree, subgraph
import torch_geometric.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


####Loading Data Set

In [3]:
import shutil
import os

# Define the path to the processed data directory
processed_dir = './data/Reddit/processed'

# Check if the directory exists and remove it to force re-download/re-processing
if os.path.exists(processed_dir):
    print(f"Removing corrupted processed data directory: {processed_dir}")
    shutil.rmtree(processed_dir)

dataset = Reddit(root='./data/Reddit', transform=T.NormalizeFeatures())
data = dataset[0].to(device)
print(f'Nodes: {data.num_nodes}')
print(f'Edges: {data.num_edges}')
print(f'Classes: {dataset.num_classes}')
print(f'Train nodes: {data.train_mask.sum()}')
print(f'Test nodes: {data.test_mask.sum()}')

Removing corrupted processed data directory: ./data/Reddit/processed


Processing...
Done!


Nodes: 232965
Edges: 114615892
Classes: 41
Train nodes: 153431
Test nodes: 55703


### Define SGC Model

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import add_self_loops, degree

def precompute(data, K=1):
    #add self loops adjacency matrix
    edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)

    # get Matrix(D)
    row, col = edge_index
    deg = degree(col, data.num_nodes, dtype=data.x.dtype)
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0 # Handle nodes with zero degree

    # edge weights for normalized adjacency (D^-0.5 * A_hat * D^-0.5)
    edge_weight = deg_inv_sqrt[row] * deg_inv_sqrt[col]

    # make a sparse adjacency tensor
    adj = torch.sparse_coo_tensor(edge_index, edge_weight, (data.num_nodes, data.num_nodes)).to(data.x.device)

    # PROPOGATE features K times using  normalize adj mtrix
    x_propagated = data.x
    for _ in range(K):
        x_propagated = torch.spmm(adj, x_propagated)
    return x_propagated

class SGC(nn.Module):
    def __init__(self, nfeat, nclass):
        super(SGC, self).__init__()
        self.lin = nn.Linear(nfeat, nclass)

    def forward(self, x):
        return self.lin(x)

## F1 Experiment Over 10 runs

In [5]:
SEEDS = list(range(10))
EPOCHS = 10
f1_scores = []
train_times = []

# precompute ONCE outside the loop
print('Precomputing train subgraph features...')
train_edge_index, _ = subgraph(
    data.train_mask, data.edge_index,
    relabel_nodes=False, num_nodes=data.num_nodes
)
train_data = data.clone()
train_data.edge_index = train_edge_index

train_prop = precompute(train_data, K=2)
train_prop = train_prop - train_prop.mean(dim=0)
train_prop = train_prop / (train_prop.std(dim=0) + 1e-8)
train_features = train_prop[data.train_mask].to(device)
train_labels   = data.y[data.train_mask].to(device)

print('Precomputing full graph features...')
full_prop = precompute(data, K=2)
full_prop = full_prop - full_prop.mean(dim=0)
full_prop = full_prop / (full_prop.std(dim=0) + 1e-8)
test_features = full_prop[data.test_mask].to(device)
test_labels   = data.y[data.test_mask].to(device)
nfeat = train_features.shape[1]
print('Precompute done. Running 10 seeds...')

# seed loop — only model init and training
for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SGC(nfeat=nfeat, nclass=dataset.num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.LBFGS(model.parameters(), lr=0.2)

    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        def closure():
            optimizer.zero_grad()
            loss = criterion(model(train_features), train_labels)
            loss.backward()
            return loss
        optimizer.step(closure)
    if device.type == 'cuda': torch.cuda.synchronize()
    train_time = time.perf_counter() - t0

    model.eval()
    with torch.no_grad():
        pred = model(test_features).argmax(dim=1).cpu().numpy()
        true = test_labels.cpu().numpy()
    f1 = f1_score(true, pred, average='micro') * 100

    f1_scores.append(f1)
    train_times.append(train_time)
    print(f'Seed {seed}: F1={f1:.2f}%  time={train_time:.3f}s')

Precomputing train subgraph features...
Precomputing full graph features...
Precompute done. Running 10 seeds...
Seed 0: F1=75.20%  time=111.215s
Seed 1: F1=77.76%  time=113.829s
Seed 2: F1=75.56%  time=113.673s
Seed 3: F1=71.55%  time=108.752s
Seed 4: F1=71.47%  time=114.183s
Seed 5: F1=72.59%  time=110.113s
Seed 6: F1=69.39%  time=109.761s
Seed 7: F1=74.23%  time=109.804s
Seed 8: F1=69.15%  time=111.442s
Seed 9: F1=70.43%  time=109.086s


In [6]:
print(f'\nSGC Reddit Results (10 seeds)')
print(f'Mean F1:   {np.mean(f1_scores):.2f} ± {np.std(f1_scores):.2f}%')
print(f'Paper F1:  94.9%')
print(f'Mean time: {np.mean(train_times):.2f}s')
print(f'Note: GCN = OOM on Reddit ')


SGC Reddit Results (10 seeds)
Mean F1:   72.73 ± 2.72%
Paper F1:  94.9%
Mean time: 111.19s
Note: GCN = OOM on Reddit (27GB requested, 14.6GB available)


In [7]:
# import torch
# import numpy as np
# import matplotlib.pyplot as plt
# from torch_geometric.datasets import Planetoid
# from torch_geometric.utils import add_self_loops, degree
# import torch_geometric.transforms as T
# from sklearn.decomposition import PCA

# dataset = Planetoid(root='./data', name='Cora', transform=T.NormalizeFeatures())
# data = dataset[0]

# def precompute(data, K=2):
#     edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
#     row, col = edge_index
#     deg = degree(col, data.num_nodes, dtype=data.x.dtype)
#     deg_inv_sqrt = deg.pow(-0.5)
#     deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
#     edge_weight = deg_inv_sqrt[row] * deg_inv_sqrt[col]
#     adj = torch.sparse_coo_tensor(edge_index, edge_weight,
#                                    (data.num_nodes, data.num_nodes))
#     x = data.x
#     for _ in range(K):
#         x = torch.spmm(adj, x)
#     return x

# x_raw      = data.x.numpy()
# x_smoothed = precompute(data, K=2).detach().numpy()
# labels     = data.y.numpy()

# from sklearn.manifold import TSNE
# raw_2d      = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(x_raw)
# smoothed_2d = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(x_smoothed)

# class_names = ['Theory','RL','Genetic Alg.','Neural Nets',
#                'Prob. Methods','Case Based','Rule Learning']
# colors = ['#E24B4A','#378ADD','#1D9E75','#EF9F27','#D4537E','#7F77DD','#639922']

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# fig.patch.set_facecolor('white')

# for ax, feats, title in zip(axes,
#                              [raw_2d, smoothed_2d],
#                              ['Raw features  (before S²X)',
#                               'Smoothed features  (after S²X)']):
#     for c, (name, color) in enumerate(zip(class_names, colors)):
#         mask = labels == c
#         ax.scatter(feats[mask, 0], feats[mask, 1],
#                    c=color, label=name, s=4, alpha=0.6, linewidths=0)
#     ax.set_title(title, fontsize=15, fontweight='bold', pad=12)
#     ax.set_xticks([]); ax.set_yticks([])
#     for spine in ax.spines.values():
#         spine.set_visible(False)

# handles = [plt.Line2D([0],[0], marker='o', color='w',
#            markerfacecolor=c, markersize=9, label=n)
#            for n, c in zip(class_names, colors)]
# fig.legend(handles=handles, loc='lower center', ncol=7,
#            fontsize=10, frameon=False,
#            bbox_to_anchor=(0.5, -0.04))

# fig.suptitle('PCA of Cora node features — S²X smoothing separates classes',  ##wait i dont think this si PCA isnt this a t SNE
#              fontsize=14, y=1.01)

# plt.tight_layout()
# plt.savefig('pca_before_after.png', dpi=200, bbox_inches='tight',
#             facecolor='white')
# plt.show()
# print('Saved pca_before_after.png')